# Adaptive Quality-Controlled cWGAN-GP for Long-Tailed Facial Expression Recognition

**Research Project:** GAN-Based Synthetic Data Augmentation for Improving Model Generalization

This notebook provides a complete GPU-accelerated execution environment for the extended **1,000-Epoch cWGAN-GP training**, automated atomic checkpointing, 4-stage adaptive quality filtering, and downstream ResNet-18 evaluation.

In [ ]:
# 1. Environment & GPU Audit
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')


In [ ]:
# 2. Mount Google Drive and Navigate to Project Directory
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
WORKSPACE = '/content/drive/MyDrive/GAN'
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)
print('Current Workspace:', os.getcwd())


In [ ]:
# 3. Install Required Dependencies
!pip install -q torchvision matplotlib pandas scipy imagehash PyWavelets PyYAML tqdm


In [ ]:
# 4. Launch Extended 1,000-Epoch cWGAN-GP Training (With Automatic Resume)
from src.training.train_wgan_gp import train_cwgan_gp

result = train_cwgan_gp(
    manifest_path='data/splits/train_manifest.csv',
    zip_path='data/raw/archive.zip',
    experiment_id='cwgan_gp_1000ep',
    epochs=1000,
    batch_size=64,
    resume=True,
    sample_interval=5,
    checkpoint_interval=1,
    historical_interval=50
)
print('Training Finished! Final Epoch:', result['last_epoch'])


In [ ]:
# 5. Execute 4-Stage Adaptive Quality Filtering on Minority Classes
from src.models.generator import ConditionalGenerator
from src.models.critic import ConditionalCritic
from src.models.resnet_classifier import ResNetClassifier
from src.data.dataset import get_dataloader
from src.training.resume import load_checkpoint
from src.filtering.adaptive_filter import AdaptiveQualityFilter
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
netG = ConditionalGenerator(100, 9, 100, feature_maps=64).to(device)
netD = ConditionalCritic(9, 100, feature_maps=64, image_size=64).to(device)
feature_ext = ResNetClassifier(num_classes=9, pretrained=True).to(device)

ckpt = 'experiments/cwgan_gp_1000ep/checkpoints/latest.pt'
optG = optim.Adam(netG.parameters(), 0.0001)
optD = optim.Adam(netD.parameters(), 0.0001)
load_checkpoint(ckpt, netG, netD, optG, optD, device)

val_loader = get_dataloader('data/splits/valid_manifest.csv', batch_size=64, shuffle=False)
train_loader = get_dataloader('data/splits/train_manifest.csv', batch_size=64, shuffle=False)

filter_engine = AdaptiveQualityFilter(netG, netD, feature_ext, latent_dim=100, device=device)
filter_engine.calibrate(val_loader, train_loader)
stats_df = filter_engine.execute_quality_filtering(
    minority_classes=[1, 2, 7],  # contempt, disgust, sleepy
    candidates_per_class=500,
    target_per_class=150
)
print(stats_df)


In [ ]:
# 6. Evaluate Downstream ResNet-18 Classifier on Held-Out Test Set
from src.evaluation.comparison import run_comparative_benchmarks

comp_df = run_comparative_benchmarks(
    train_manifest='data/splits/train_manifest.csv',
    val_manifest='data/splits/valid_manifest.csv',
    test_manifest='data/splits/test_manifest.csv',
    synthetic_manifest='data/synthetic/synthetic_manifest.csv',
    epochs_per_run=25
)
print(comp_df)
